# ECG Heartbeat Data Exploration

Exploratory analysis of the MIT-BIH Arrhythmia dataset used by this project.
We inspect the class distribution (which is heavily imbalanced) and visualise
representative beats for each AAMI category.

> Run this notebook from the project root, or it will adjust `sys.path` itself.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))  # make `src` importable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import TRAIN_CSV, N_FEATURES, CLASS_NAMES

df = pd.read_csv(TRAIN_CSV, header=None)
print('Shape:', df.shape)
df.head()

## Class distribution

The dataset is dominated by Normal beats. This is why training applies SMOTE
to oversample the minority arrhythmia classes.

In [ ]:
labels = df.iloc[:, N_FEATURES].astype(int)
counts = labels.value_counts().sort_index()
counts.index = [CLASS_NAMES[i] for i in counts.index]

ax = counts.plot(kind='bar', figsize=(8, 4), color='steelblue')
ax.set_title('Heartbeat class distribution (training set)')
ax.set_ylabel('Number of beats')
for i, v in enumerate(counts.values):
    ax.text(i, v, f'{v:,}', ha='center', va='bottom')
plt.tight_layout(); plt.show()
counts

## Representative beats per class

Each row is a single heartbeat, normalised to [0, 1] and zero-padded to 187
samples. Different arrhythmias have visibly different morphologies.

In [ ]:
fig, axes = plt.subplots(1, len(CLASS_NAMES), figsize=(18, 3), sharey=True)
for cls, ax in enumerate(axes):
    sample = df[df.iloc[:, N_FEATURES] == cls].iloc[0, :N_FEATURES].values
    ax.plot(sample, color='crimson')
    ax.set_title(CLASS_NAMES[cls])
    ax.set_xlabel('time step')
axes[0].set_ylabel('amplitude')
plt.suptitle('Example heartbeat per class', y=1.05)
plt.tight_layout(); plt.show()

## Mean waveform per class

Averaging many beats reveals the characteristic shape of each category and how
much the QRS complex differs between them.

In [ ]:
plt.figure(figsize=(10, 5))
for cls in range(len(CLASS_NAMES)):
    mean_beat = df[df.iloc[:, N_FEATURES] == cls].iloc[:, :N_FEATURES].mean().values
    plt.plot(mean_beat, label=CLASS_NAMES[cls])
plt.title('Mean heartbeat waveform per class')
plt.xlabel('time step'); plt.ylabel('mean amplitude'); plt.legend()
plt.tight_layout(); plt.show()